In [0]:
# ══════════════════════════════════════
# EXPLORACAO — Analise das Tabelas
# Squad 3 — Batch Ecommerce
# Objetivo: entender os dados antes de
# montar a arquitetura medalhao
# ══════════════════════════════════════

# Celula 1 — Carrega config e utils
%run ./config/00_config.ipynb
%run ./utils/00_utils.ipynb


In [0]:
# Celula 2 — Funcao de Analise
def analisar_tabela(df, nome):
    print("=" * 55)
    print(f"ANALISE EXPLORATORIA — {nome}")
    print("=" * 55)

    print(f"\nShape: {df.shape}")
    print(f"   {df.shape[0]} linhas | {df.shape[1]} colunas")

    print("\nColunas e tipos:")
    print(df.dtypes)

    print("\nNulos por coluna:")
    print(df.isnull().sum())

    print(f"\nDuplicatas: {df.duplicated().sum()}")

    print("\nEstatisticas:")
    print(df.describe())

    print("\nPrimeiras 5 linhas:")
    print(df.head())

    print("\nUltimas 5 linhas:")
    print(df.tail())

print("Funcao analisar_tabela criada!")


In [0]:
# Celula 3 — Analisar physical_lojas (suas tabela 1)
print("Lendo physical_lojas...")
df_lojas = ler_csv(
    adls_client,
    container,
    "batch-data/physical_lojas.csv"
)
analisar_tabela(df_lojas, "physical_lojas")


In [0]:
# Celula 4 — Analisar physical_itens_venda_caixa (sua tabela 2 — 277MB)
print("Lendo physical_itens_venda_caixa em chunks...")
df_itens = ler_csv_chunks(
    adls_client,
    container,
    "batch-data/physical_itens_venda_caixa.csv",
    chunk_size=50000
)
analisar_tabela(df_itens, "physical_itens_venda_caixa")


In [0]:
# Celula 5 — Analisar physical_vendas_caixa
# Apenas para entender o id_transacao e verificar se tem data
# NAO e sua tabela — usada so como referencia
print("Lendo physical_vendas_caixa (apenas referencia)...")
df_vendas = ler_csv(
    adls_client,
    container,
    "batch-data/physical_vendas_caixa.csv"
)

print("=" * 55)
print("ANALISE DE REFERENCIA — physical_vendas_caixa")
print("(Tabela NAO pertence ao seu escopo)")
print("=" * 55)

print(f"\nShape: {df_vendas.shape}")
print(f"   {df_vendas.shape[0]} linhas | {df_vendas.shape[1]} colunas")

print("\nColunas e tipos:")
print(df_vendas.dtypes)

print("\nPrimeiras 5 linhas:")
print(df_vendas.head())

In [0]:
# Celula 6 — Verificar id_transacao
print("=" * 55)
print("VERIFICACAO — id_transacao")
print("=" * 55)

print("\nAmostra id_transacao em physical_itens_venda_caixa:")
print(df_itens['id_transacao'].head(10).to_string())

print("\nTamanho dos valores:")
print(df_itens['id_transacao'].str.len().value_counts())

if 'id_transacao' in df_vendas.columns:
    print("\nAmostra id_transacao em physical_vendas_caixa:")
    print(df_vendas['id_transacao'].head(10).to_string())

In [0]:
# Celula 7 — Verificar se id_transacao tem data embutida
print("=" * 55)
print("VERIFICACAO — Data embutida no id_transacao")
print("=" * 55)

# Tenta extrair data do id_transacao
amostra = df_itens['id_transacao'].head(20)
print("\nAmostra completa para analise de padrao:")
for val in amostra:
    print(f"   {val}")



In [0]:
# Celula 8 — Verificar relacionamento FK
print("=" * 55)
print("RELACIONAMENTO — id_transacao")
print("=" * 55)

if 'id_transacao' in df_vendas.columns:
    ids_vendas = set(df_vendas['id_transacao'].astype(str).unique())
    ids_itens  = set(df_itens['id_transacao'].astype(str).unique())
    orfaos     = ids_itens - ids_vendas

    print(f"\nTransacoes unicas em itens        : {len(ids_itens):,}")
    print(f"Transacoes unicas em vendas       : {len(ids_vendas):,}")
    print(f"Itens orfaos (sem venda pai)      : {len(orfaos):,}")

    if len(orfaos) > 0:
        print(f"\nExemplos de orfaos:")
        for orfao in list(orfaos)[:5]:
            print(f"   - {orfao}")

# Verificar id_loja em vendas vs lojas
if 'id_loja' in df_vendas.columns:
    ids_lojas_cadastro = set(df_lojas['id_loja'].astype(str).unique())
    ids_lojas_vendas   = set(df_vendas['id_loja'].astype(str).unique())
    lojas_sem_cadastro = ids_lojas_vendas - ids_lojas_cadastro

    print(f"\nLojas unicas em vendas            : {len(ids_lojas_vendas):,}")
    print(f"Lojas unicas no cadastro          : {len(ids_lojas_cadastro):,}")
    print(f"Lojas em vendas sem cadastro      : {len(lojas_sem_cadastro):,}")